# The SIDM file census

This notebook shows how to find out **which input ROOT files of a sample are actually good** —
which open and read, which are corrupt, which are unreachable, and which are *empty* (0 events) —
and how to turn that into clean per-sample file lists that jobs run over.

## Why this exists

`make_fileset` only ever sees the files that survive `yaml.safe_load` of a location YAML. The
files that someone hand-commented-out (`# - ...root`) are **invisible** to it — so a file that was
retired months ago for a transient reason is never re-checked, and a genuinely corrupt file that
slipped into the live list is only discovered when a job crashes on it. On top of that:

- a **corrupt** file can `stat` fine and even *open*, yet have a truncated `Runs` tree (you can't
  reconstruct more events than were generated) — a plain existence check misses this;
- an **empty** file (0 events survived the skim) is perfectly valid, but a Condor chunk built
  *entirely* of empties makes coffea raise `TypeError: cannot unpack non-iterable NoneType object`;
- **unreachable** files are silently dropped by `skipbadfiles`, quietly biasing any normalization
  that divides by a sum of generator weights.

The census opens **every** file (commented ones included), gives each a durable verdict, and rolls
the verdicts into a self-describing manifest + cleaned filelists.

## The verdicts

| verdict | meaning |
|---|---|
| `reachable` | opens and reads — the analysis can use it. `events_entries == 0` ⇒ also flagged **empty** |
| `bad` | opens but is corrupt/truncated (e.g. `Events > genEventCount` ⇒ truncated `Runs`), or a deterministic decompress error |
| `unreachable` | the server positively reports "no such file" |
| `inconclusive` | could not be determined (timeout / transient) — re-probed next run, **never** blacklisted |

The discipline (shared with the Condor campaign reconciler) is: **condemn a file only on a
positive, deterministic failure.** A timeout or redirector hiccup is `inconclusive`, not `bad`.

**Two depths.** *Shallow* (the default, and what we publish) opens each file and reads only its
per-file `Runs` tree + peeks `Events` — no event loop. *Deep* (`--deep`) additionally runs the full
`SidmProcessor` on each reachable file to certify it processes; it is expensive and is used as an
occasional validation, not the published light census.

## 1. Load a published census

Every census this analysis has run is indexed in [`sidm/configs/census/README.md`](../configs/census/README.md); each run's full manifest + cleaned filelists live on EOS under `/store/group/lpcmetx/SIDM/census/<run-id>/`. The published run-ids are:

| run-id | what |
|---|---|
| `signal_2mu2e_v10_light` | 2mu2e signal (4082 files) |
| `signal_4mu_v10_light` | 4mu signal (4960 files) |
| `backgrounds_skimmed_light` | skimmed backgrounds (56296 files) |
| `data_skimmed_light` | skimmed data (127415 files) |
| `data_light` | data (636 files) |

Below we read the 2mu2e census as the worked example — swap the run-id in the URL to load any other.

In [1]:
import sys, os, json, subprocess, tempfile
import pandas as pd

# Make the SIDM package importable whether this notebook is launched from the repo root or
# from its own directory (sidm/studies/file_census/).
REPO = next((os.path.abspath(up) for up in [".", "..", "../..", "../../..", "../../../.."]
             if os.path.isdir(os.path.join(up, "sidm", "tools"))), ".")
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("repo root:", REPO)

PUB = "root://cmseos.fnal.gov//store/group/lpcmetx/SIDM/census/signal_2mu2e_v10_light/manifest.json"

# coffea/json read local files only, so xrdcp the manifest down first, then json.load it.
tmp = os.path.join(tempfile.gettempdir(), "census_2mu2e_manifest.json")
subprocess.run(["xrdcp", "-f", "-s", PUB, tmp], check=True)
manifest = json.load(open(tmp))
print("schema_version:", manifest["schema_version"], "| run_id:", manifest["run_id"])
print("files:", len(manifest["files"]), "| samples:", len(manifest["samples"]))

repo root: /uscms_data/d3/murtazas/SIDM-wt-file-census
schema_version: 2 | run_id: signal_2mu2e_v10_light
files: 4082 | samples: 90


## 2. Provenance — `meta`

Every manifest is **self-describing**: when it was produced, from which YAML (and its sha256, so you know exactly what it corresponds to), by whom, on what, with which uproot/sidm versions. `complete` + `n_enumerated`/`n_probed` tell you it is a *full* census, not a partial one.

In [2]:
meta = manifest["meta"]
for k in ["mode", "complete", "n_enumerated", "n_probed", "source_yaml", "source_yaml_sha256",
          "version", "started_utc", "ended_utc", "duration_s", "user", "host",
          "uproot_version", "sidm_commit", "redirector"]:
    print(f"{k:20s} {meta.get(k)}")

mode                 shallow
complete             True
n_enumerated         4082
n_probed             4082
source_yaml          /uscms_data/d3/murtazas/SIDM-wt-file-census/sidm/configs/ntuples/signal_2mu2e_v10.yaml
source_yaml_sha256   7655da76702596e531bf038e1c4af9c2012fa99221ff1d29206296477e49ce9f
version              llpNanoAOD_v2
started_utc          2026-06-23T06:08:24Z
ended_utc            2026-06-23T06:45:42Z
duration_s           2238.0
user                 murtazas
host                 cmslpc306.fnal.gov
uproot_version       5.7.4
sidm_commit          a8f82f0
redirector           root://cmseos.fnal.gov


## 3. The per-sample rollup

One row per sample. **How to read it:** `reachable`/`empty`/`bad`/`unreachable` are file counts; `skim_basis` is the MEASURED pre/post-skim verdict (`no_runs` = skimmed sample whose `Runs` tree was stripped, so its normalization comes from `skim_factor` + an `Events` genWeight sum, not from the file); `genEventSumw_reachable_norm` is the sum of generator weights over the reachable files, already corrected for the skim where that applies.

In [3]:
samples = pd.DataFrame(manifest["samples"])
cols = ["sample", "n_files", "reachable", "n_empty", "bad", "unreachable", "inconclusive",
        "skim_basis", "events_over_runs_ratio", "genEventSumw_reachable_norm"]
view = samples[[c for c in cols if c in samples.columns]]
print(f"{len(view)} samples; showing the first 12")
view.head(12)

90 samples; showing the first 12


,sample,n_files,reachable,n_empty,bad,unreachable,inconclusive,skim_basis,events_over_runs_ratio,genEventSumw_reachable_norm
0,2Mu2E_100GeV_0p25GeV_0p02mm,44,43,0,1,0,0,gen_filtered,0.0965,1300517.0
1,2Mu2E_100GeV_0p25GeV_0p2mm,42,41,0,1,0,0,gen_filtered,0.0971,1183115.0
2,2Mu2E_100GeV_0p25GeV_2p0mm,47,46,0,1,0,0,gen_filtered,0.0987,1248448.0
3,2Mu2E_100GeV_0p25GeV_10p0mm,45,42,0,3,0,0,gen_filtered,0.0968,1080510.0
4,2Mu2E_100GeV_0p25GeV_20p0mm,45,44,0,1,0,0,gen_filtered,0.0970,817484.0
5,2Mu2E_100GeV_1p2GeV_0p096mm,46,46,0,0,0,0,gen_filtered,0.0952,1087414.0
6,2Mu2E_100GeV_1p2GeV_0p96mm,46,45,0,1,0,0,gen_filtered,0.0955,1038686.0
7,2Mu2E_100GeV_1p2GeV_9p6mm,47,45,0,2,0,0,gen_filtered,0.0961,1022127.0
8,2Mu2E_100GeV_1p2GeV_48p0mm,47,46,0,1,0,0,gen_filtered,0.0963,917487.0
9,2Mu2E_100GeV_1p2GeV_96p0mm,46,45,0,1,0,0,gen_filtered,0.0960,644378.0


## 4. The actionable output — files to act on

The whole point: the specific **bad** and **empty** files. Bad files are corruption a plain open-check would miss; empty files are valid but must not fill a whole Condor chunk.

In [4]:
bad = [r for r in manifest["files"] if r["status"] in ("bad", "unreachable")]
empty = [r for r in manifest["files"]
         if r["status"] == "reachable" and (r["events_entries"] or 0) == 0]
print(f"{len(bad)} bad/unreachable, {len(empty)} empty\n")
for r in bad[:6]:
    print(f"  [{r['status']}] {r['sample']}/{r['filename']}")
    print(f"       {r['last_error'][:90]}")
print(f"  ... ({len(bad)} bad/unreachable total)")

59 bad/unreachable, 3 empty

  [bad] 2Mu2E_100GeV_0p25GeV_0p02mm/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-100_MDp-0p25_ctau-0p02_v3_part-27.root
       truncated/corrupt Runs: Events 284 > genEventCount 141
  [bad] 2Mu2E_100GeV_0p25GeV_0p2mm/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-100_MDp-0p25_ctau-0p2_v3_part-41.root
       truncated/corrupt Runs: Events 297 > genEventCount 154
  [bad] 2Mu2E_100GeV_0p25GeV_10p0mm/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-100_MDp-0p25_ctau-10p0_v3_part-1.root
       truncated/corrupt Runs: Events 270 > genEventCount 135
  [bad] 2Mu2E_100GeV_0p25GeV_10p0mm/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-100_MDp-0p25_ctau-10p0_v3_part-13.root
       truncated/corrupt Runs: Events 269 > genEventCount 137
  [bad] 2Mu2E_100GeV_0p25GeV_10p0mm/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-100_MDp-0p25_ctau-10p0_v3_part-22.root
       truncated/corrupt Runs: Events 269 > genEventCount 130
  [bad] 2Mu2E_100GeV_0p25GeV_20p0mm/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-100_MDp-0p25_ctau-20p0_v3

A commented-out file that actually **opens fine** is a candidate to un-comment (it was retired for a transient reason). The census surfaces these too:

In [5]:
revived = [r for r in manifest["files"]
           if r["source"] == "commented" and r["status"] == "reachable"
           and (r["events_entries"] or 0) > 0]
print(f"{len(revived)} commented-but-good files (candidates to restore)")
for r in revived[:5]:
    print(f"  {r['sample']}/{r['filename']}  (YAML note: {r['commented_reason'] or '-'})")

129 commented-but-good files (candidates to restore)
  2Mu2E_100GeV_0p25GeV_2p0mm/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-100_MDp-0p25_ctau-2p0_v3_part-7.root  (YAML note: LZMA Corrupt input data)
  2Mu2E_100GeV_5p0GeV_0p4mm/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-100_MDp-5p0_ctau-0p4_v3_part-40.root  (YAML note: LZMA Corrupt input data)
  2Mu2E_100GeV_5p0GeV_200mm/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-100_MDp-5p0_ctau-200_v3_part-35.root  (YAML note: LZMA Corrupt input data)
  2Mu2E_150GeV_5p0GeV_2p7mm/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-150_MDp-5p0_ctau-2p7_v3_part-7.root  (YAML note: LZMA Corrupt input data)
  2Mu2E_200GeV_0p25GeV_0p01mm/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-200_MDp-0p25_ctau-0p01_v3_part-0.root  (YAML note: LZMA Corrupt input data)


## GenPart self-reference (the `distinctParent` hang)

Some skimmed QCD outputs carry corrupt **self-referential** `GenPart` entries (a particle whose
`genPartIdxMother` points to its own index). coffea's `distinctParent` climbs the same-pdgId
mother chain in an unguarded loop, so that self-loop never terminates and the **job hangs** on a
single such event. The census reads `GenPart_genPartIdxMother` on every MC file **by default**
(`--no-check-genpart` to disable) and records `n_genpart_selfref_events` per file (the **GENPART
INTEGRITY** report section). These files are **kept** (`reachable`) — the hang itself is fixed by
the cycle-safe kernel in `sidm/tools/gen.py`, installed via the LLPNanoAOD schema. List the
flagged files in a manifest with:

```python
genpart = [r for r in manifest["files"] if r.get("n_genpart_selfref_events")]
```

(The signal census loaded above is clean — the corruption lives in the QCD skims; load
`backgrounds_skimmed` to see flagged files.)

## 5. Cleaned filelists — what jobs run over

`filelists` turns a manifest into one `<sample>.txt` of the **good** files (reachable, non-empty — live *or* commented-but-good, so a wrongly-retired file is restored), plus `_dropped.tsv` (with reasons) and a `_census_ref.json` provenance pointer. Jobs then point at these lists instead of the raw YAML.

In [6]:
from sidm.tools import file_census as fc

out_dir = os.path.join(tempfile.gettempdir(), "cleaned_demo")
summary = fc.cleaned_filelists(manifest, out_dir)
print(summary)
print("\nfiles written:", sorted(os.listdir(out_dir))[:6], "...")
one = next(f for f in os.listdir(out_dir) if f.endswith(".txt"))
print(f"\nhead of {one}:")
print("\n".join(open(os.path.join(out_dir, one)).read().splitlines()[:2]))

{'n_samples': 90, 'n_good': 4020, 'n_dropped': 62, 'out_dir': '/tmp/cleaned_demo'}

files written: ['2Mu2E_1000GeV_0p25GeV_0p002mm.txt', '2Mu2E_1000GeV_0p25GeV_0p02mm.txt', '2Mu2E_1000GeV_0p25GeV_0p2mm.txt', '2Mu2E_1000GeV_0p25GeV_1p0mm.txt', '2Mu2E_1000GeV_0p25GeV_2p0mm.txt', '2Mu2E_1000GeV_1p2GeV_0p0096mm.txt'] ...

head of 2Mu2E_150GeV_1p2GeV_6p4mm.txt:
root://cmseos.fnal.gov//store/group/lpcmetx/SIDM/ULSignalSamples/2018_v10/BsTo2DpTo2Mu2e/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-150_MDp-1p2_ctau-6p4_v3/LLPnanoAODv2/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-150_MDp-1p2_ctau-6p4_v3_part-0.root
root://cmseos.fnal.gov//store/group/lpcmetx/SIDM/ULSignalSamples/2018_v10/BsTo2DpTo2Mu2e/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-150_MDp-1p2_ctau-6p4_v3/LLPnanoAODv2/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-150_MDp-1p2_ctau-6p4_v3_part-1.root


## Skip flagged files in your jobs (the veto)

`filelists` also writes `_skip.json` — the per-sample list of flagged files (bad / unreachable /
empty; plus the self-reference files with `--drop-genpart-corrupt`). `make_fileset` consumes it
to drop those files **in addition to** the ones the location YAML comments out (the YAML's manual
commenting is always honored):

```python
from sidm.tools import utilities
fileset = utilities.make_fileset(samples, version, location_cfg="backgrounds.yaml",
                                 census_skip="backgrounds_skimmed.skip.json")
```

For Condor: `python condor/make_job_args.py … --census-skip backgrounds_skimmed.skip.json`.
`census_skip` is a path or a bare name resolved under `sidm/configs/census/`; matching is by
basename per sample.

## 6. Running the census yourself

### Locally / threaded (from the submitter node)

The shallow probe is I/O-bound, so threads parallelize the unavoidable per-file open timeouts. Good for one sample or a quick check:

```bash
python sidm/tools/file_census.py census --location-cfg signal_2mu2e_v10.yaml \
    --version llpNanoAOD_v2 --workers 16 --run-id signal_2mu2e_v10 \
    --out census_signal_2mu2e_v10.json
```

Below we run it live over just a few files of one sample to show the real output:

In [7]:
sample = manifest["samples"][0]["sample"]
yaml_path = os.path.join(REPO, "sidm/configs/ntuples/signal_2mu2e_v10.yaml")
small = fc.run_census(yaml_path, version="llpNanoAOD_v2",
                      samples=[sample], max_live=3, workers=3, progress=False)
print("ran census on", small["meta"]["n_probed"], "files of", sample,
      "| complete:", small["meta"]["complete"])
s0 = small["samples"][0]
print(f"  {s0['sample']}: reachable={s0['reachable']} empty={s0['n_empty']} "
      f"bad={s0['bad']} skim_basis={s0['skim_basis']}")

ran census on 3 files of 2Mu2E_100GeV_0p25GeV_0p02mm | complete: True
  2Mu2E_100GeV_0p25GeV_0p02mm: reachable=3 empty=0 bad=0 skim_basis=gen_filtered


### On dask (LPC Condor workers)

For a cheap shallow census at larger scale, map the probe over batches on dask workers:

```bash
python sidm/scripts/census_dask.py --location-cfg backgrounds.yaml \
    --version skimmed_llpNanoAOD_v2 --batch-size 50 --max-workers 40 \
    --out census_backgrounds.json --run-id backgrounds
```

The driver rides through evicted workers and **refuses to publish a partial run** (it stamps
`meta.complete=false` and exits non-zero if any batch was lost).

### On Condor (the authoritative, detached run — and what scales to 100k+ files)

```bash
# 1. chunk (writes census_job_args.txt, filelists, and census_expected.json)
python condor/make_census_args.py --location-cfg backgrounds.yaml \
    --version skimmed_llpNanoAOD_v2 --files-per-job 500
# 2. rebuild the (slim) code tarball WITH the census worker + campaign_lib, and mkdir the EOS dir
tar -czf condor/sidm_code.tar.gz sidm \
    condor/run_sidm_chunk.py condor/run_census_chunk.py condor/campaign_lib.py
xrdfs root://cmseos.fnal.gov mkdir -p /store/user/$USER/sidm_census/run_v1
# 3. submit, then 4. merge WITH the expected sidecar so a missing shard is caught, not published:
python condor/merge_census_shards.py --shard-eos-dir /store/user/$USER/sidm_census/run_v1 \
    --expected condor/census_expected.json --out census_backgrounds.json --run-id backgrounds
```

`merge` reconciles the shards against `census_expected.json` and only writes `complete=true` when
every enumerated file is accounted for. `condor/CENSUS_README.md` has the full reference.

## 7. Storing outputs on shared EOS

Publish each census to the shared lpcmetx area so collaborators consume it without re-running:

```
/store/group/lpcmetx/SIDM/census/<run-id>/
    manifest.json              # the full per-file manifest
    cleaned_filelists.tar.gz   # the per-sample good-file lists jobs run over
```

and commit a small **summary** (per-sample rollup + the dropped list + the provenance `meta`) to
`sidm/configs/census/<run-id>.census.summary.json` so the result is reviewable in the PR without
the multi-MB manifest.

**Publish light, validate with deep.** The committed/published artifact is the *shallow* census —
it is cheap to regenerate whenever EOS or the YAML changes, carries the source-YAML sha256, and
catches the real failures. Run a *deep* census occasionally as validation; if it finds a file that
opens but won't process, add that file to the drop list. The heavy deep manifest is not the thing
you commit.